<a href="https://colab.research.google.com/github/yangml103/catanatron/blob/master/examples/Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Catanatron Introduction
This shows example usage of Catanatron. First, clone the repo and install requirements.

In [1]:
import os

# Only clone if not already present
if not os.path.exists('code'):
    !git clone https://github.com/bcollazo/catanatron.git code

# Uninstall dopamine-rl to avoid dependency conflicts with gymnasium
%pip uninstall -y dopamine-rl

# Install catanatron dependencies
%pip install -e code[web,gym,dev]

exit() # Forcefully restart runtime to picks up installed requirements

Found existing installation: dopamine_rl 4.1.2
Uninstalling dopamine_rl-4.1.2:
  Successfully uninstalled dopamine_rl-4.1.2
Obtaining file:///content/code
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for catanatron (pyproject.toml) ... done
  Created wheel for catanatron: filename=catanatron-3.3.0-0.editable-py3-none-any.whl size=15940 sha256=e0c5b29a469dd54344987f590ae5d13ac4cb460a346e60ee5e907ba7d6410081
  Stored in directory: /tmp/pip-ephem-wheel-cache-e7fkqwir/wheels/b3/a0/80/88e352e9f114ee791347bc788e8272f2675482a9ab6bd38f2a
Successfully built catanatron
  Attempting uninstall: catanatron
    Found existing installation: catanatron 3.3.0
    Uninstalling catanatron-3.3.0:
      Successfully uninstalled catanatron-3.3.0


Then, you can implement your own bot strategy and pit against some benchmark bots.

In [1]:
import random

from catanatron.game import Game
from catanatron.models.player import Player, RandomPlayer, Color
from catanatron.players.weighted_random import WeightedRandomPlayer

class MyPlayer(Player):
    def decide(self, game, playable_actions):
        """Should return one of the playable_actions.

        Args:
            game (Game): complete game state. read-only.
            playable_actions (Iterable[Action]): options to choose from
        Return:
            action (Action): Chosen element of playable_actions
        """
        # ===== YOUR CODE HERE =====
        # As an example we simply choose a valid action at random:
        return random.choice(playable_actions)
        # ===== END YOUR CODE =====

# Play a simple 4v4 game. Edit MyPlayer with your logic!
players = [
    MyPlayer(Color.RED),
    WeightedRandomPlayer(Color.BLUE),
    RandomPlayer(Color.WHITE),
    RandomPlayer(Color.ORANGE),
]
game = Game(players)
print(game.play())  # returns winning color

Color.WHITE


You can also simulate thousands of games to get more statistically significant results:

In [2]:
from pprint import pprint
from catanatron.cli.play import play_batch

wins, results_by_player, games = play_batch(10, players)

Output()

                                      Last 10 Games                                      
     ╷                       ╷       ╷        ╷         ╷          ╷           ╷         
   # │ SEATING               │ TURNS │ RED VP │ BLUE VP │ WHITE VP │ ORANGE VP │ WINNER  
╶────┼───────────────────────┼───────┼────────┼─────────┼──────────┼───────────┼────────╴
   1 │ ORANGE,WHITE,RED,BLUE │   433 │      9 │      10 │        7 │         5 │ BLUE    
   2 │ WHITE,RED,ORANGE,BLUE │   357 │      9 │      10 │        7 │         3 │ BLUE    
   3 │ RED,ORANGE,BLUE,WHITE │    72 │      2 │      10 │        3 │         2 │ BLUE    
   4 │ BLUE,ORANGE,RED,WHITE │   207 │      5 │       5 │        2 │        10 │ ORANGE  
   5 │ WHITE,ORANGE,RED,BLUE │   141 │      3 │      10 │        2 │         3 │ BLUE    
   6 │ BLUE,RED,ORANGE,WHITE │   295 │     10 │       5 │        3 │         5 │ RED     
   7 │ ORANGE,WHITE,BLUE,RED │   291 │      5 │       3 │       10 │         5 │ WHITE   
   8 │ BLUE,RED,WHITE,ORANGE │    94 │      2 │      10 │        2 │         2 │ BLUE    
   9 │ ORANGE,BLUE,RED,WHITE │   171 │      3 │      10 │        2 │         3 │ BLUE    
  10 │ WHITE,BLUE,RED,ORANGE │   363 │      8 │      10 │        2 │         8 │ BLUE    
     ╵                       ╵       ╵        ╵         ╵          ╵           ╵

                                              Player Summary                                               
                            ╷      ╷        ╷             ╷            ╷          ╷          ╷             
                            │ WINS │ AVG VP │ AVG SETTLES │ AVG CITIES │ AVG ROAD │ AVG ARMY │ AVG DEV VP  
╶───────────────────────────┼──────┼────────┼─────────────┼────────────┼──────────┼──────────┼────────────╴
  MyPlayer:RED              │    1 │   5.60 │        2.30 │       0.70 │     0.10 │     0.20 │       1.30  
  WeightedRandomPlayer:BLUE │    7 │   8.30 │        3.20 │       1.00 │     0.40 │     0.40 │       1.50  
  RandomPlayer:WHITE        │    1 │   4.00 │        2.30 │       0.50 │     0.20 │     0.10 │       0.10  
  RandomPlayer:ORANGE       │    1 │   4.60 │        1.80 │       0.70 │     0.10 │     0.20 │       0.80  
                            ╵      ╵        ╵             ╵            ╵          ╵          ╵

              Game Summary              
            ╷           ╷               
  AVG TICKS │ AVG TURNS │ AVG DURATION  
╶───────────┼───────────┼──────────────╴
     723.90 │    242.40 │   0.032 secs  
            ╵           ╵

You can inspect the game states in a variety of ways and compute statistics

In [3]:
from catanatron.json import GameEncoder
from catanatron.features import create_sample_vector, create_sample
from catanatron.gym.board_tensor_features import (
    create_board_tensor,
)

game = games[0]  # pick say the first one

# 1. Feature dictionary of last state before game ended from REDs perspective.
#   See https://catanatron.readthedocs.io/en/latest/catanatron.gym.envs.html#catanatron.gym.envs.catanatron_env.CatanatronEnv.observation_space
#   for more information on this representation.
# record = create_sample(game, Color.RED)
# pprint(record)

# 2. Vector (similar to 1) of last state before game ended
# vector = create_sample_vector(game, Color.RED)
# print(vector)

# 3. Board Tensor representation, similar to the one described in https://arxiv.org/abs/2008.07079
# tensor = create_board_tensor(game, Color.RED)
# print(tensor)

# 4. Inspect Python catanatron.state.State class
# print(game.state)

# 5. JSON Representation (with full action history)
game_json = GameEncoder().default(game)
pprint(game_json)  # inspect a game state representation

{'action_records': [ActionRecord(action=Action(color=C.ORANGE, action_type=AT.BUILD_SETTLEMENT, value=9), result=None),
                    ActionRecord(action=Action(color=C.ORANGE, action_type=AT.BUILD_ROAD, value=(2, 9)), result=None),
                    ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_SETTLEMENT, value=40), result=None),
                    ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_ROAD, value=(40, 44)), result=None),
                    ActionRecord(action=Action(color=C.RED, action_type=AT.BUILD_SETTLEMENT, value=7), result=None),
                    ActionRecord(action=Action(color=C.RED, action_type=AT.BUILD_ROAD, value=(7, 8)), result=None),
                    ActionRecord(action=Action(color=C.BLUE, action_type=AT.BUILD_SETTLEMENT, value=4), result=None),
                    ActionRecord(action=Action(color=C.BLUE, action_type=AT.BUILD_ROAD, value=(3, 4)), result=None),
                    ActionRecord(action=Action(color=

# Training

In [5]:
pip install sb3-contrib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 22.3 MB/s eta 0:00:00


In [10]:
import gymnasium as gym
import numpy as np
import catanatron.gym  # Registers 'catanatron/Catanatron-v0'
import torch
# Disable strict "Simplex" checking to prevent crashes from float precision errors
torch.distributions.Distribution.set_default_validate_args(False)
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def mask_fn(env: gym.Env) -> np.ndarray:
    """
    Callback to return a boolean mask of valid actions for the current state.
    True = Action is valid
    False = Action is invalid
    """
    # catanatron.gym environments provide a .get_valid_actions() method.
    # We access .unwrapped to ensure we reach the base environment
    # regardless of other wrappers (like ActionMasker or Monitor).
    valid_action_indices = env.unwrapped.get_valid_actions()

    # Create a boolean mask of all False (invalid)
    mask = np.zeros(env.action_space.n, dtype=bool)

    # Set valid indices to True
    mask[valid_action_indices] = True

    return mask

def main():
    # 1. Initialize the Environment
    # The environment is registered as 'catanatron/Catanatron-v0'
    env = gym.make("catanatron/Catanatron-v0")

    # 2. Wrap Env with ActionMasker
    # This wrapper allows MaskablePPO to query our mask_fn during training
    # to purely focus on valid moves.
    env = ActionMasker(env, mask_fn)

    # 3. Initialize the Agent
    model = MaskablePPO(
        MaskableActorCriticPolicy,
        env,
        device=device,
        verbose=1,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=0.995,   # High discount factor because Catan is a long game
        ent_coef=0.01  # Entropy coefficient to encourage exploration
    )

    # 4. Train
    print("Starting training...")
    # Competitive bots often need millions of steps.
    model.learn(total_timesteps=100_000_000)
    print("Training complete.")

    # 5. Save
    model.save("ppo_catanatron_agent")

    # 6. Evaluation Loop
    # NOTE: When running inference (predict), you MUST pass the mask manually.
    obs, _ = env.reset()
    done = False

    print("\nPlaying a test game...")
    while not done:
        # Calculate mask for the current state
        action_masks = mask_fn(env)

        # Pass action_masks to predict() to ensure the agent picks a legal move
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)

        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    # Check who won (reward is usually +1 for win, -1 for loss)
    print(f"Game finished. Reward: {reward}")

if __name__ == "__main__":
    main()

Using device: cuda
Vectorizing environment across 4 CPU cores...


ConnectionResetError: [Errno 104] Connection reset by peer

## Parallelized Training BROKEN

In [ ]:
import gymnasium as gym
import numpy as np
import catanatron.gym  # Registers 'catanatron/Catanatron-v0'
import torch
import multiprocessing
# Disable strict "Simplex" checking to prevent crashes from float precision errors
torch.distributions.Distribution.set_default_validate_args(False)

from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def mask_fn(env: gym.Env) -> np.ndarray:
    """
    Callback to return a boolean mask of valid actions for the current state.
    True = Action is valid
    False = Action is invalid
    """
    valid_action_indices = env.unwrapped.get_valid_actions()
    mask = np.zeros(env.action_space.n, dtype=bool)
    mask[valid_action_indices] = True
    return mask

def main():
    # 1. Initialize Vectorized Environment
    # Use all available CPU cores to run games in parallel
    n_envs = multiprocessing.cpu_count()
    print(f"Vectorizing environment across {n_envs} CPU cores...")

    # We use make_vec_env with SubprocVecEnv for true parallelism.
    # We must pass the wrapper class so each subprocess wraps its own env.
    env = make_vec_env(
        "catanatron/Catanatron-v0",
        n_envs=n_envs,
        vec_env_cls=SubprocVecEnv,
        wrapper_class=ActionMasker,
        wrapper_kwargs={"action_mask_fn": mask_fn}
    )

    # 2. Initialize the Agent
    model = MaskablePPO(
        MaskableActorCriticPolicy,
        env,
        device=device,
        verbose=1,
        learning_rate=3e-4,
        n_steps=2048, # Steps per environment before update
        batch_size=256, # Increased batch size for more stable GPU updates
        gamma=0.995,
        ent_coef=0.01
    )

    # 3. Train
    print("Starting training...")
    # Total timesteps will be distributed across all envs
    model.learn(total_timesteps=100_000)
    print("Training complete.")

    # 4. Save
    model.save("ppo_catanatron_agent")

    # Close the training envs to free up resources
    env.close()

    # 5. Evaluation Loop
    # For evaluation, we use a SINGLE environment to play one coherent game.
    print("\nPlaying a test game (Single Env)...")
    eval_env = gym.make("catanatron/Catanatron-v0")
    eval_env = ActionMasker(eval_env, mask_fn)

    obs, _ = eval_env.reset()
    done = False

    while not done:
        # Calculate mask for the current state
        action_masks = mask_fn(eval_env)

        # Pass action_masks to predict() to ensure the agent picks a legal move
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)

        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated

    print(f"Game finished. Reward: {reward}")
    eval_env.close()

if __name__ == "__main__":
    main()